# CIFAR-10 CNN Experiments — Task 1 (Parts A, B, C)

Run this on **Colab with a GPU** (Runtime → Change runtime type → GPU). Each experiment appends one row to `results.csv`, and the confusion matrix / training history for each run is saved so you can pull numbers straight into the report.

Fixed for the whole notebook: seed = 42, 90/10 train/val split, 20-epoch training budget, batch size 128.

In [ ]:
import os, csv, time, json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

print(tf.__version__)
print(tf.config.list_physical_devices())

## 1. Seed, data, fixed split

This cell is run once. Everything downstream reuses `x_train`, `x_val`, `x_test` defined here — don't reload or reshuffle later, or experiments stop being comparable.

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

EPOCHS_BUDGET = 20          # fixed training budget for every experiment (Part A's budget)
BATCH_SIZE = 128
CLASS_NAMES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]

(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train_full, y_test = y_train_full.flatten(), y_test.flatten()
x_train_full, x_test = x_train_full.astype("float32") / 255.0, x_test.astype("float32") / 255.0

# frozen 90/10 train/val split, same indices every time (seeded shuffle)
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(x_train_full))
val_n = int(0.1 * len(idx))
val_idx, train_idx = idx[:val_n], idx[val_n:]
x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]
x_val, y_val = x_train_full[val_idx], y_train_full[val_idx]

print(x_train.shape, x_val.shape, x_test.shape)

RESULTS_CSV = "results.csv"
if os.path.isfile(RESULTS_CSV):
    os.remove(RESULTS_CSV)  # fresh run

## 2. Model builder

AlexNet-style, adapted for 32×32 input (filters scaled down from the original 96/256/384, fewer pooling stages so spatial dims don't collapse to 0). All the Part B knobs (dropout, batchnorm, L2, extra block, kernel size) are parameters on this one function so nothing is duplicated per-experiment.

In [ ]:
def build_model(filters=(64, 128, 256), dropout=0.0, batchnorm=False, l2=0.0,
                 extra_block=False, kernel_size=3):
    reg = regularizers.l2(l2) if l2 > 0 else None
    inp = layers.Input(shape=(32, 32, 3))
    x = inp
    blocks = list(filters) + ([filters[-1] * 2] if extra_block else [])
    for f in blocks:
        x = layers.Conv2D(f, kernel_size, padding="same", activation="relu", kernel_regularizer=reg)(x)
        if batchnorm:
            x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D(2)(x)
        if dropout > 0:
            x = layers.Dropout(dropout)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(512, activation="relu", kernel_regularizer=reg)(x)
    if batchnorm:
        x = layers.BatchNormalization()(x)
    if dropout > 0:
        x = layers.Dropout(dropout)(x)
    out = layers.Dense(10, activation="softmax")(x)
    return models.Model(inp, out)

build_model().summary()

## 3. Reusable train + eval routine

Every experiment in Parts A, B, C calls this one function. It logs one row to `results.csv`, and saves the confusion matrix (`cm_<name>.npy`) and training history (`history_<name>.json`) per run.

In [ ]:
def train_and_eval(name, model, optimizer="adam", lr=1e-3, epochs=EPOCHS_BUDGET,
                    augment=None, log=True):
    tf.random.set_seed(SEED)
    opt_map = {
        "adam": tf.keras.optimizers.Adam(learning_rate=lr),
        "sgd": tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9),
        "rmsprop": tf.keras.optimizers.RMSprop(learning_rate=lr),
    }
    model.compile(optimizer=opt_map[optimizer], loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])

    if augment:
        datagen = tf.keras.preprocessing.image.ImageDataGenerator(**augment)
        datagen.fit(x_train)
        train_flow = datagen.flow(x_train, y_train, batch_size=BATCH_SIZE, seed=SEED)
        t0 = time.time()
        hist = model.fit(train_flow, validation_data=(x_val, y_val),
                          epochs=epochs, verbose=2)
    else:
        t0 = time.time()
        hist = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                          epochs=epochs, batch_size=BATCH_SIZE, verbose=2)
    train_time = time.time() - t0

    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    preds = np.argmax(model.predict(x_test, verbose=0), axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, preds, average="macro")
    cm = confusion_matrix(y_test, preds)
    n_params = model.count_params()

    row = dict(
        name=name, epochs=epochs, train_time_sec=round(train_time, 1),
        n_params=n_params,
        train_acc=round(hist.history["accuracy"][-1], 4),
        val_acc=round(hist.history["val_accuracy"][-1], 4),
        test_acc=round(test_acc, 4),
        precision=round(precision, 4), recall=round(recall, 4), f1=round(f1, 4),
    )
    if log:
        write_row(row)
    np.save(f"cm_{name}.npy", cm)
    with open(f"history_{name}.json", "w") as f:
        json.dump(hist.history, f)
    print(row)
    return row, cm


def write_row(row):
    exists = os.path.isfile(RESULTS_CSV)
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            w.writeheader()
        w.writerow(row)

## Part A — Baseline

No dropout, no batch norm, no L2, no augmentation — a clean baseline every later experiment is measured against. Target: >70% test accuracy.

In [ ]:
baseline_row, baseline_cm = train_and_eval("baseline", build_model(), optimizer="adam", lr=1e-3)

### Confusion matrix (baseline)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_cm(cm, title):
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(title)
    plt.tight_layout(); plt.show()

plot_cm(baseline_cm, "Baseline confusion matrix")

# most-confused off-diagonal pairs
cm_off = baseline_cm.copy()
np.fill_diagonal(cm_off, 0)
top_pairs = np.dstack(np.unravel_index(np.argsort(-cm_off, axis=None), cm_off.shape))[0][:6]
for i, j in top_pairs:
    print(f"{CLASS_NAMES[i]} -> {CLASS_NAMES[j]}: {cm_off[i][j]}")

## Part B — Controlled Experiments

Same seed, same split, same 20-epoch budget as Part A in every cell below — only one variable changes per experiment.

### B1. Regularization study

In [ ]:
train_and_eval("reg_dropout", build_model(dropout=0.3))
train_and_eval("reg_batchnorm", build_model(batchnorm=True))
train_and_eval("reg_l2", build_model(l2=1e-4))

### B2. Data augmentation study

In [ ]:
light = dict(horizontal_flip=True)
moderate = dict(horizontal_flip=True, rotation_range=10, width_shift_range=0.1, height_shift_range=0.1)
aggressive = dict(horizontal_flip=True, rotation_range=30, zoom_range=0.3,
                   brightness_range=(0.6, 1.4), width_shift_range=0.2, height_shift_range=0.2)

train_and_eval("aug_light", build_model(), augment=light)
train_and_eval("aug_moderate", build_model(), augment=moderate)
train_and_eval("aug_aggressive", build_model(), augment=aggressive)

### B3. Optimization study

In [ ]:
for opt in ["sgd", "adam", "rmsprop"]:
    train_and_eval(f"opt_{opt}", build_model(), optimizer=opt, lr=1e-3)

for lr in [1e-2, 1e-3, 1e-4]:
    train_and_eval(f"lr_{lr}", build_model(), optimizer="adam", lr=lr)

### B4. Architecture study

In [ ]:
train_and_eval("arch_extra_block", build_model(extra_block=True))
train_and_eval("arch_kernel5", build_model(kernel_size=5))

## Part C — Final Customized CNN

Fill in `best_*` below with whichever techniques actually won in Part B — don't just turn everything on. Look at `results.csv` from the cells above before running this.

In [ ]:
def part_c(best_dropout=0.3, best_batchnorm=True, best_l2=0.0,
           best_optimizer="adam", best_lr=1e-3, best_augment=None, best_extra_block=False):
    model = build_model(dropout=best_dropout, batchnorm=best_batchnorm, l2=best_l2,
                         extra_block=best_extra_block)
    row, cm = train_and_eval("final_custom", model, optimizer=best_optimizer,
                              lr=best_lr, augment=best_augment)
    return row, cm

# EDIT the arguments above to match your actual Part B winners, then run:
final_row, final_cm = part_c()

### Final comparison: baseline vs. custom CNN

In [ ]:
plot_cm(final_cm, "Final custom model — confusion matrix")

gain = final_row["test_acc"] - baseline_row["test_acc"]
extra_params_m = (final_row["n_params"] - baseline_row["n_params"]) / 1e6
print(f"Test accuracy: baseline {baseline_row['test_acc']*100:.2f}%  ->  custom {final_row['test_acc']*100:.2f}%  ({gain*100:+.2f} pts)")
print(f"Params: baseline {baseline_row['n_params']:,}  ->  custom {final_row['n_params']:,}  ({extra_params_m:+.2f}M)")
print(f"Training time: baseline {baseline_row['train_time_sec']/60:.1f} min  ->  custom {final_row['train_time_sec']/60:.1f} min")
if extra_params_m != 0:
    print(f"Accuracy gained per extra million params: {(gain*100)/extra_params_m:.2f} pts/M")

## 4. Results table

Everything logged across Parts A, B, C, in one place — this is what you copy into the report.

In [ ]:
import pandas as pd
df = pd.read_csv("results.csv")
df